# BirdCLEF+ 2026 - Google Perch v2 Probe

Purpose: extract frozen Perch embeddings and train a shallow PyTorch probe.  
Artifacts are written to `/kaggle/working/artifacts/perch_v2`.


## 1. Setup


In [2]:
from pathlib import Path
import json
import os
import random
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)


class CFG:
    seed = 42
    competition_name = "birdclef-2026"
    data_root = None
    artifact_dir = Path("/kaggle/working/artifacts")


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)


def find_data_root() -> Path:
    candidates = [
        Path("/kaggle/input/birdclef-2026"),
        Path("/kaggle/input/birdclef-2026-repack/birdclef-2026"),
        Path("/kaggle/input/birdclef-2026-repack"),
        Path("data/raw/birdclef-2026"),
    ]
    for path in candidates:
        if (path / "train.csv").exists():
            return path
    input_root = Path("/kaggle/input")
    if input_root.exists():
        matches = list(input_root.glob("**/train.csv"))
        if matches:
            return matches[0].parent
    raise FileNotFoundError("Could not find train.csv. Attach the BirdCLEF 2026 dataset.")


def read_optional_csv(path: Path) -> pd.DataFrame | None:
    return pd.read_csv(path) if path.exists() else None


seed_everything(CFG.seed)
CFG.data_root = find_data_root()
CFG.artifact_dir.mkdir(parents=True, exist_ok=True)

print(f"Data root: {CFG.data_root}")
print(f"Artifacts: {CFG.artifact_dir}")

import librosa
from sklearn.model_selection import train_test_split
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

try:
    import tensorflow as tf
except ImportError:
    tf = None

torch.manual_seed(CFG.seed)
torch.cuda.manual_seed_all(CFG.seed)


class CFG(CFG):
    artifact_dir = Path("/kaggle/working/artifacts/perch_v2")
    sample_rate = 32000
    duration = 5.0
    embedding_dim = 1536
    extraction_batch_size = 8
    probe_batch_size = 128
    probe_epochs = 10
    hidden_dim = 512
    dropout = 0.25
    lr = 1e-3
    max_samples = None
    perch_model_dir = None


CFG.artifact_dir.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


Data root: /kaggle/input/competitions/birdclef-2026
Artifacts: /kaggle/working/artifacts
Device: cuda


## 2. Load Metadata


In [3]:
train = pd.read_csv(CFG.data_root / "train.csv")
train["filepath"] = train["filename"].map(lambda x: CFG.data_root / "train_audio" / x)
if CFG.max_samples:
    train = train.sample(CFG.max_samples, random_state=CFG.seed).reset_index(drop=True)

labels = sorted(train["primary_label"].unique())
label_to_idx = {label: idx for idx, label in enumerate(labels)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}
train["target"] = train["primary_label"].map(label_to_idx)
(CFG.artifact_dir / "labels.json").write_text(json.dumps(idx_to_label, indent=2), encoding="utf-8")

print(f"Rows: {len(train):,}")
print(f"Classes: {len(labels):,}")
display(train.head())


Rows: 35,549
Classes: 206


,primary_label,secondary_labels,type,latitude,longitude,scientific_name,common_name,class_name,inat_taxon_id,author,license,rating,url,filename,collection,filepath,target
0,1161364,[],[],-22.7562,-46.8666,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1216197....,1161364/iNat1216197.ogg,iNat,/kaggle/input/competitions/birdclef-2026/train...,0
1,1161364,[],[],-22.7558,-46.8700,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1114648....,1161364/iNat1114648.ogg,iNat,/kaggle/input/competitions/birdclef-2026/train...,0
2,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/810195.m...,1161364/iNat810195.ogg,iNat,/kaggle/input/competitions/birdclef-2026/train...,0
3,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/818781.m...,1161364/iNat818781.ogg,iNat,/kaggle/input/competitions/birdclef-2026/train...,0
4,1161364,[],[],-22.7426,-46.8985,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/556514.m...,1161364/iNat556514.ogg,iNat,/kaggle/input/competitions/birdclef-2026/train...,0


## 3. Locate And Load Perch


In [4]:
def find_perch_model_dir() -> Path:
    if CFG.perch_model_dir:
        return Path(CFG.perch_model_dir)
    input_root = Path("/kaggle/input")
    matches = list(input_root.glob("**/saved_model.pb")) if input_root.exists() else []
    matches = [path.parent for path in matches if "perch" in str(path).lower() or "vocal" in str(path).lower()]
    if matches:
        return matches[0]
    raise FileNotFoundError(
        "Could not find a Perch SavedModel. Attach the Perch/vocalization-classifier Kaggle model "
        "or set CFG.perch_model_dir."
    )


if tf is None:
    raise ImportError("TensorFlow is required for Perch embedding extraction.")

perch_model_dir = find_perch_model_dir()
perch = tf.saved_model.load(str(perch_model_dir))
infer = perch.signatures["serving_default"]
print(f"Perch model: {perch_model_dir}")
print(f"Inputs: {infer.structured_input_signature}")
print(f"Outputs: {infer.structured_outputs}")


I0000 00:00:1778317055.117573      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778317055.123414      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
2026-05-09 08:57:35.373018: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_shardy_partitioner which is not in the op definition: Op<name=XlaCallModule; signature=args: -> output:; attr=version:int; attr=module:string; attr=Sout:list(shape),min=0; attr=Tout:list(type),min=0; attr=Tin:list(type),min=0; attr=dim_args_spec:list(string),default=[]; attr=platforms:list(string),default=[]; attr=function_list:list(func),default=[]; attr=has_token_input_output:bool,default=false; attr=disabled_checks:list(string),default=[]; is

Perch model: /kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2/2
Inputs: ((), {'inputs': TensorSpec(shape=(None, 160000), dtype=tf.float32, name='inputs')})
Outputs: {'embedding': TensorSpec(shape=(None, 1536), dtype=tf.float32, name='embedding'), 'spatial_embedding': TensorSpec(shape=(None, 16, 4, 1536), dtype=tf.float32, name='spatial_embedding'), 'spectrogram': TensorSpec(shape=(None, 500, 128), dtype=tf.float32, name='spectrogram'), 'label': TensorSpec(shape=(None, 14795), dtype=tf.float32, name='label')}


## 4. Extract Embeddings


In [5]:
def load_audio(path: Path) -> np.ndarray:
    target_len = int(CFG.sample_rate * CFG.duration)
    y, _ = librosa.load(path, sr=CFG.sample_rate, mono=True, duration=CFG.duration)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    return y[:target_len].astype(np.float32)


def run_perch_batch(batch_waveforms: np.ndarray) -> np.ndarray:
    tensor = tf.convert_to_tensor(batch_waveforms, dtype=tf.float32)
    _, keyword_specs = infer.structured_input_signature
    if keyword_specs:
        input_name = next(iter(keyword_specs))
        outputs = infer(**{input_name: tensor})
    else:
        outputs = infer(tensor)
    value = next(iter(outputs.values()))
    return np.asarray(value).astype(np.float32)


embeddings_path = CFG.artifact_dir / "train_embeddings.npz"
if embeddings_path.exists():
    saved = np.load(embeddings_path, allow_pickle=True)
    embeddings = saved["embeddings"]
else:
    chunks = []
    waveforms = []
    for path in tqdm(train["filepath"], desc="audio"):
        waveforms.append(load_audio(path))
        if len(waveforms) == CFG.extraction_batch_size:
            chunks.append(run_perch_batch(np.stack(waveforms)))
            waveforms = []
    if waveforms:
        chunks.append(run_perch_batch(np.stack(waveforms)))
    embeddings = np.concatenate(chunks, axis=0)
    np.savez_compressed(
        embeddings_path,
        embeddings=embeddings,
        labels=train["primary_label"].to_numpy(),
        filenames=train["filename"].to_numpy(),
    )

print(f"Embeddings: {embeddings.shape}")
print(f"Saved: {embeddings_path}")


audio:   0%|          | 0/35549 [00:00<?, ?it/s]

I0000 00:00:1778317072.021863     132 service.cc:152] XLA service 0x7cc428073ac0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778317072.021923     132 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1778317072.021927     132 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
2026-05-09 08:57:52.050089: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_shardy_partitioner which is not in the op definition: Op<name=XlaCallModule; signature=args: -> output:; attr=version:int; attr=module:string; attr=Sout:list(shape),min=0; attr=Tout:list(type),min=0; attr=Tin:list(type),min=0; attr=dim_args_spec:list(string),default=[]; attr=platforms:list(string),default=[]; attr=function_list:list(func),default=[]; attr=has_token_input_output:bool,default=false; attr=disabled_checks:list(string),default=[]; is_stateful=true> This may be expected if y

InvalidArgumentError: Graph execution error:

Detected at node XlaCallModule defined at (most recent call last):
<stack traces unavailable>
XlaCallModuleOp with version 10 is not supported by this build. Must be <= 9
	 [[{{node XlaCallModule}}]]
	tf2xla conversion failed while converting __inference_predict_fn_5016[]. Run with TF_DUMP_GRAPH_PREFIX=/path/to/dump/dir and --vmodule=xla_compiler=2 to obtain a dump of the compiled functions.
	 [[StatefulPartitionedCall]] [Op:__inference_signature_wrapper_5524]

## 5. Probe Model


In [ ]:
class PerchProbe(nn.Module):
    def __init__(self, embedding_dim: int, num_classes: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(embedding_dim),
            nn.Linear(embedding_dim, CFG.hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(CFG.dropout),
            nn.Linear(CFG.hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.net(x)


x = embeddings.astype(np.float32)
y = train["target"].to_numpy(dtype=np.int64)
train_idx, valid_idx = train_test_split(
    np.arange(len(y)),
    test_size=0.2,
    random_state=CFG.seed,
    stratify=y,
)

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(x[train_idx]), torch.from_numpy(y[train_idx])),
    batch_size=CFG.probe_batch_size,
    shuffle=True,
)
valid_loader = DataLoader(
    TensorDataset(torch.from_numpy(x[valid_idx]), torch.from_numpy(y[valid_idx])),
    batch_size=CFG.probe_batch_size * 2,
    shuffle=False,
)

model = PerchProbe(embedding_dim=x.shape[1], num_classes=len(labels)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr)


## 6. Train And Save Artifacts


In [ ]:
@torch.no_grad()
def validate() -> float:
    model.eval()
    correct = 0
    seen = 0
    for xb, yb in valid_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        logits = model(xb)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        seen += xb.size(0)
    return correct / max(seen, 1)


history = []
best_acc = 0.0
for epoch in range(1, CFG.probe_epochs + 1):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)

    valid_acc = validate()
    row = {
        "epoch": epoch,
        "train_loss": total_loss / max(len(train_loader.dataset), 1),
        "valid_acc": valid_acc,
    }
    history.append(row)
    print(row)
    if valid_acc > best_acc:
        best_acc = valid_acc
        torch.save(
            {
                "model": model.state_dict(),
                "label_to_idx": label_to_idx,
                "cfg": {k: v for k, v in CFG.__dict__.items() if not k.startswith("_")},
                "valid_acc": best_acc,
            },
            CFG.artifact_dir / "best_perch_probe.pt",
        )

pd.DataFrame(history).to_csv(CFG.artifact_dir / "history.csv", index=False)
print(f"Best valid accuracy: {best_acc:.4f}")
print(f"Artifacts saved to {CFG.artifact_dir}")


## 7. Package Artifacts For Download


This cell zips the extracted Perch embeddings, label map, probe checkpoint, and training history. The embeddings file can be large, but keeping it in the zip is useful when you want to train probes locally without rerunning TensorFlow extraction.


In [ ]:
import zipfile
from IPython.display import FileLink

zip_path = Path("/kaggle/working/birdclef_perch_v2_artifacts.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(CFG.artifact_dir.rglob("*")):
        if path.is_file():
            zf.write(path, arcname=path.relative_to(CFG.artifact_dir.parent))

print(f"Wrote {zip_path} ({zip_path.stat().st_size / 1024 / 1024:.2f} MB)")
display(FileLink(zip_path))
